In [ ]:
!pip install -q torch-geometric

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.signal import hilbert, welch
from scipy.stats import skew, kurtosis, entropy
from torch_geometric.utils import dropout_edge
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, auc,
                             precision_recall_curve, average_precision_score)
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool
import joblib
import json

sns.set(style='whitegrid')

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
     torch.cuda.manual_seed(RANDOM_STATE)  

# ==============================================================================
# MODEL
# ==============================================================================

class ProposedGraphSAGE(nn.Module):
    """
    3-layer GraphSAGE with LayerNorm, ReLU, and dropout.
    Global mean pooling → MLP classifier.
    """
    def __init__(self, in_feats, hidden=64, num_classes=2, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_feats, hidden)
        self.ln1   = nn.LayerNorm(hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.ln2   = nn.LayerNorm(hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.ln3   = nn.LayerNorm(hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_classes)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.ln1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.ln2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.ln3(self.conv3(x, edge_index)))
        x = global_mean_pool(x, batch)
        return self.classifier(x)


# ==============================================================================
# AUGMENTATION  ← applied to TRAINING subjects ONLY
# ==============================================================================

# Suffixes used to tag each augmented copy
AUG_SUFFIXES = ('_orig', '_noise', '_shift')

def augment_add_noise(ts_df, sigma=0.05):
    """Add zero-mean Gaussian noise to every ROI signal."""
    return pd.DataFrame(
        ts_df.values + np.random.normal(0, sigma, ts_df.shape),
        columns=ts_df.columns
    )

def augment_time_shift(ts_df, shift=10):
    """Circularly shift the time-series along the temporal axis."""
    shifted = np.roll(ts_df.values, shift=shift, axis=0)
    return pd.DataFrame(shifted, columns=ts_df.columns)

def build_augmented_train(train_sids, ts_data, labels_dict):
    """
    Create 3 augmented copies for each training subject:
      <sid>_orig  → original (no change)
      <sid>_noise → original + Gaussian noise
      <sid>_shift → original with circular time shift

    Val and test subjects are NOT passed here and stay unmodified.

    Returns
    -------
    aug_ts     : {augmented_sid: DataFrame}
    aug_labels : {augmented_sid: int}
    aug_sids   : ordered list of all augmented train SIDs
    """
    aug_ts, aug_labels = {}, {}

    for sid in train_sids:
        df  = ts_data[sid]
        lab = labels_dict[sid]

        aug_ts[f"{sid}_orig"]  = df.copy()
        aug_ts[f"{sid}_noise"] = augment_add_noise(df)
        aug_ts[f"{sid}_shift"] = augment_time_shift(df)

        aug_labels[f"{sid}_orig"]  = lab
        aug_labels[f"{sid}_noise"] = lab
        aug_labels[f"{sid}_shift"] = lab

    aug_sids = list(aug_ts.keys())
    return aug_ts, aug_labels, aug_sids

def get_base_sid(sid):
    """Strip the augmentation suffix to get the original subject ID."""
    for suffix in AUG_SUFFIXES:
        if sid.endswith(suffix):
            return sid[:-len(suffix)]
    return sid


# ==============================================================================
# FEATURE EXTRACTION
# ==============================================================================

def extract_node_features(signal, fs=0.5):
    """
    Compute a 13-dim feature vector for one ROI signal:
    mean, std, skewness, kurtosis, RMS, energy, zero-crossings,
    spectral entropy, spectral centroid, wavelet energy (db4 lv3),
    lag-1 autocorrelation, Q1, Q3.
    """
    feats  = []
    signal = np.asarray(signal).ravel()

    feats.append(np.mean(signal))
    feats.append(np.std(signal))
    feats.append(skew(signal))
    feats.append(kurtosis(signal))
    feats.append(np.sqrt(np.mean(signal ** 2)))                          # RMS
    feats.append(np.sum(signal ** 2))                                    # energy
    feats.append(int(np.sum(np.abs(np.diff(np.sign(signal))) > 0)))      # zero-crossings

    freqs, psd = welch(signal, fs=fs, nperseg=min(64, len(signal)))
    psd_sum    = np.sum(psd)
    psd_norm   = psd / psd_sum if psd_sum > 0 else psd
    feats.append(entropy(psd_norm) if psd_sum > 0 else 0)               # spectral entropy
    feats.append(np.sum(freqs * psd_norm) / np.sum(psd_norm)
                 if psd_sum > 0 else 0)                                  # spectral centroid

    try:
        coeffs = pywt.wavedec(signal, 'db4', level=3)
        feats.append(np.sum(np.square(coeffs[-1])) if len(coeffs) > 0 else 0)
    except Exception:
        feats.append(0)

    feats.append(np.corrcoef(signal[:-1], signal[1:])[0, 1]
                 if len(signal) > 1 else 0)                             # lag-1 autocorr
    feats.append(np.percentile(signal, 25))
    feats.append(np.percentile(signal, 75))

    return np.array(feats, dtype=float)


# ==============================================================================
# GRAPH CONSTRUCTION  (PLV connectivity)
# ==============================================================================

def compute_phase(signal):
    """Instantaneous phase via Hilbert transform."""
    return np.angle(hilbert(signal))

def compute_plv(sig1, sig2):
    """Phase-Locking Value between two ROI signals."""
    d = compute_phase(sig1) - compute_phase(sig2)
    return float(np.abs(np.mean(np.exp(1j * d))))

def generate_plv_matrix(df):
    """Full symmetric PLV matrix for all ROI pairs of a subject."""
    n   = df.shape[1]
    mat = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(i, n):
            val       = compute_plv(df.iloc[:, i].values, df.iloc[:, j].values)
            mat[i, j] = val
            mat[j, i] = val
    return mat

def plv_to_adj(plv_mat, threshold=0.5):
    """
    Binary adjacency matrix from PLV: keep edges with PLV >= threshold.
    Diagonal (self-loops) is set to 0.
    """
    adj = (np.abs(plv_mat) >= threshold).astype(int)
    np.fill_diagonal(adj, 0)
    return adj

def build_gcn_data_list(sids, node_features_dict, adj_dict, labels_dict, scaler=None):
    """Convert raw features + adjacency matrices into a list of PyG Data objects."""
    data_list = []
    for sid in sids:
        node_feats = node_features_dict[sid]
        x = torch.tensor(
            scaler.transform(node_feats) if scaler is not None else node_feats,
            dtype=torch.float
        )
        adj        = adj_dict[sid]
        rows, cols = np.where(adj > 0)

        # Fall back to self-loops if graph is fully disconnected
        if rows.size == 0:
            rows = np.arange(x.shape[0])
            cols = np.arange(x.shape[0])

        edge_index = torch.tensor(
            np.vstack([np.concatenate([rows, cols]),
                       np.concatenate([cols, rows])]),
            dtype=torch.long
        )
        y = labels_dict[sid]
        data_list.append(
            Data(x=x, edge_index=edge_index, y=torch.tensor([y], dtype=torch.long))
        )
    return data_list


# ==============================================================================
# THRESHOLD SELECTION
# ==============================================================================

def select_best_threshold_variant(y_true, y_prob, min_recall=0.75):
    """
    Sweep thresholds [0.01 … 0.99] and pick the one that maximises
    precision subject to recall >= min_recall.
    """
    best_th, best_score = 0.5, -np.inf
    for th in np.linspace(0.01, 0.99, 99):
        preds = (np.array(y_prob) >= th).astype(int)
        prec  = precision_score(y_true, preds, zero_division=0)
        rec   = recall_score(y_true, preds, zero_division=0)
        score = prec if rec >= min_recall else -np.inf
        if score > best_score:
            best_score = score
            best_th    = th
    return best_th, best_score


# ==============================================================================
# TRAINING & VALIDATION
# ==============================================================================

def train_and_evaluate_gcn(
    train_node_features_dict, train_adj_dict, train_labels_dict,
    val_node_features_dict,   val_adj_dict,   val_labels_dict, fold_num=None,
    n_epochs=100, batch_size=16, lr=1e-3,
    class_weight_multiplier=3.0,
    threshold_mode='precision_at_min_recall',
    min_recall_target=0.75
):
    """
    Train ProposedGraphSAGE on the (augmented) training set and
    evaluate on the validation set.

    Returns val metrics, best model, scaler, epoch info, and val predictions.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    train_sids = list(train_node_features_dict.keys())
    val_sids   = list(val_node_features_dict.keys())

    # Fit scaler on training features only (no val/test leakage)
    all_x_train = np.vstack([train_node_features_dict[s] for s in train_sids])
    scaler      = StandardScaler().fit(all_x_train)

    train_data = build_gcn_data_list(
        train_sids, train_node_features_dict, train_adj_dict, train_labels_dict, scaler
    )
    val_data = build_gcn_data_list(
        val_sids, val_node_features_dict, val_adj_dict, val_labels_dict, scaler
    )

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=batch_size, shuffle=False)

    # Class-weighted loss — extra boost for the minority class
    train_labels   = np.array([train_labels_dict[s] for s in train_sids])
    unique_classes = np.unique(train_labels)
    cw             = list(compute_class_weight('balanced', classes=unique_classes, y=train_labels))
    if len(cw) == 2:
        counts = [(train_labels == c).sum() for c in unique_classes]
        cw[int(np.argmin(counts))] *= class_weight_multiplier
    class_weights = torch.tensor(cw, dtype=torch.float).to(device)

    in_feats  = train_data[0].x.shape[1]
    model     = ProposedGraphSAGE(in_feats=in_feats, hidden=64, num_classes=2).to(device)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-3)  # weight_decay=1e-3 is used for individual site datasets (e.g., KKI, NeuroImage)
                                                                    # weight_decay=1e-5 is used for the combined dataset in the reported experiments

    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_val_loss    = float('inf')
    best_model_state = None
    best_epoch_info  = {}
    patience_counter = 0
    patience         = 15 # use 15 for small/individual sites (e.g., KKI, NeuroImage), 30 for combined dataset
    min_delta        = 0.001

    desc = f"Fold {fold_num} - Training" if fold_num is not None else "Training"
    pbar = tqdm(range(n_epochs), desc=desc, leave=True, dynamic_ncols=True, 
                bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]")

    for epoch in pbar:
    
        # Training
        # ── Training step ──────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out  = model(batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

         

        # ── Validation step ────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        y_true_v, y_prob_v = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out   = model(batch)
                loss  = criterion(out, batch.y)
                val_loss += loss.item()
                probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
                y_prob_v.extend(probs)
                y_true_v.extend(batch.y.cpu().numpy())
        val_loss /= len(val_loader)
        val_acc   = accuracy_score(y_true_v, (np.array(y_prob_v) >= 0.5).astype(int))

        scheduler.step(val_loss)


        # ── Early Stopping ─────────────────────────────────────────────────────
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                pbar.set_postfix({"Train Loss": f"{train_loss:.4f}", 
                                  "Val Loss": f"{val_loss:.4f}", 
                                  "Early Stop": f"epoch {epoch+1}"})
                break

        # Mise à jour de la barre avec Train Loss + Val Loss
        pbar.set_postfix({"Train Loss": f"{train_loss:.4f}", 
                          "Val Loss": f"{val_loss:.4f}"})
    # ── Reload best checkpoint → final val predictions ─────────────────────────
    model.load_state_dict(best_model_state)
    model.eval()
    y_true_val, y_prob_val = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out   = model(batch)
            probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
            y_prob_val.extend(probs)
            y_true_val.extend(batch.y.cpu().numpy())

    best_th, _ = select_best_threshold_variant(y_true_val, y_prob_val, min_recall=min_recall_target)
    y_pred_val  = (np.array(y_prob_val) >= best_th).astype(int)

    metrics = {
        'accuracy':  accuracy_score(y_true_val, y_pred_val),
        'precision': precision_score(y_true_val, y_pred_val, zero_division=0),
        'recall':    recall_score(y_true_val, y_pred_val, zero_division=0),
        'f1':        f1_score(y_true_val, y_pred_val, zero_division=0),
        'roc_auc':   roc_auc_score(y_true_val, y_prob_val)
    }

    return metrics, model, scaler, best_epoch_info, y_true_val, y_pred_val, y_prob_val


# ==============================================================================
# CONFIGURATION
# ==============================================================================


TS_NPY      = "/kaggle/input/datasets/timeseries_NeuroIMAGE.npy"
LAB_NPY     = "/kaggle/input/datasets/labels_NeuroIMAGE.npy"
RESULTS_DIR = "/kaggle/working/Results-FULL"


os.makedirs(RESULTS_DIR, exist_ok=True)

N_SPLITS               = 5
N_EPOCHS               = 100  # 100 for individual sites; 200 for combined dataset
BATCH_SIZE             = 16
LR                     = 1e-3
CLASS_WEIGHT_MULTIPLIER = 3
PLV_PERCENTILE         = 80


# ==============================================================================
# DATA LOADING
# ==============================================================================

_ts_arr  = np.load(TS_NPY)
_lab_arr = np.load(LAB_NPY).astype(int)
assert len(_ts_arr) == len(_lab_arr), "Mismatch in number of subjects!"

# Ensure shape is (N, T, R)  — swap axes if needed
if _ts_arr.shape[1] < _ts_arr.shape[2]:
    _ts_arr = _ts_arr.transpose(0, 2, 1)

ts_data, labels_dict = {}, {}
for i, (ts, lab) in enumerate(zip(_ts_arr, _lab_arr)):
    sid              = f"subj_{i:04d}"
    ts_data[sid]     = pd.DataFrame(ts)
    labels_dict[sid] = int(lab)

print(f"Subjects loaded : {len(ts_data)}")
print(f"TS shape/subject: {_ts_arr.shape[1:]} (T×R)")
n0 = int(np.sum(_lab_arr == 0))
n1 = int(np.sum(_lab_arr == 1))
print(f"  → Class 0 (control): {n0} | Class 1 (ADHD): {n1} | Ratio: {n0/max(n1,1):.2f}")


# ==============================================================================
# PRE-COMPUTATION  (raw features + PLV matrices on the full dataset)
# NOTE: PLV *threshold* is derived from training data only, inside the fold loop.
# ==============================================================================

node_features_raw, plv_matrices = {}, {}
for sid, df in tqdm(ts_data.items(), desc="Initial feature extraction"):
    node_features_raw[sid] = np.vstack(
        [extract_node_features(df.iloc[:, j].values) for j in range(df.shape[1])]
    )
    plv_matrices[sid] = generate_plv_matrix(df)

subject_ids = list(ts_data.keys())
y           = np.array([labels_dict[sid] for sid in subject_ids])


# ==============================================================================
# NESTED CROSS-VALIDATION LOOP  (5 outer × 4 inner)
# ==============================================================================

all_metrics                     = []
y_true_all, y_pred_all, y_prob_all = [], [], []
best_model, best_scaler, best_epoch_info = None, None, None
best_f1 = 0.0

skf_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (trainval_idx, test_idx) in enumerate(skf_outer.split(subject_ids, y)):

    test_sids     = [subject_ids[i] for i in test_idx]
    trainval_sids = [subject_ids[i] for i in trainval_idx]
    y_trainval    = np.array([labels_dict[sid] for sid in trainval_sids])

    # Single inner split: 3/4 train, 1/4 validation
    skf_inner = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
    inner_train_idx, inner_val_idx = next(skf_inner.split(trainval_sids, y_trainval))

    train_sids = [trainval_sids[i] for i in inner_train_idx]
    val_sids   = [trainval_sids[i] for i in inner_val_idx]

    print(f"  Test set : {len(test_sids)} subjects")

    # ==========================================================================
    # AUGMENTATION — training subjects ONLY
    # Each train subject gets 3 copies: _orig, _noise, _shift
    # Val and test subjects are added as-is (no modification)
    # ==========================================================================
    aug_ts_data, aug_labels, aug_train_sids = build_augmented_train(
        train_sids, ts_data, labels_dict
    )

    # Add val and test originals into the lookup dict (no augmentation)
    for sid in val_sids + test_sids:
        aug_ts_data[sid] = ts_data[sid]
        aug_labels[sid]  = labels_dict[sid]

    # Safety check: no augmented base-SID should overlap with val or test
    aug_base_set = {get_base_sid(s) for s in aug_train_sids}
    assert aug_base_set.isdisjoint(set(val_sids)),  "❌ Augmentation leakage into val!"
    assert aug_base_set.isdisjoint(set(test_sids)), "❌ Augmentation leakage into test!"
    print(f"  ✅ No augmentation leakage detected")
    print(f"  Augmented train size: {len(aug_train_sids)} "
          f"({len(train_sids)} subjects × 3 copies)")

    # ==========================================================================
    # PLV THRESHOLD — computed from ORIGINAL training subjects only
    # (augmented copies are excluded to avoid any leakage bias)
    # ==========================================================================
    train_offdiag = []
    for sid in train_sids:                        # original sids, not augmented
        mat = plv_matrices[sid]
        idx = np.triu_indices(mat.shape[0], k=1)
        train_offdiag.append(mat[idx])
    train_offdiag  = np.concatenate(train_offdiag)
    fold_threshold = np.percentile(train_offdiag, PLV_PERCENTILE)
    print(f"🔧 PLV threshold (fold {fold+1}): {fold_threshold:.4f}")

    # ==========================================================================
    # FEATURE & GRAPH CONSTRUCTION
    # Process augmented train + original val + original test
    # ==========================================================================
    adj_matrices           = {}
    combined_node_features = {}

    # Gather all SIDs to process in one pass
    all_fold_sids = aug_train_sids + val_sids + test_sids

    for sid in tqdm(all_fold_sids, desc="Processing data"):
        df         = aug_ts_data[sid]
        node_feats = np.vstack(
            [extract_node_features(df.iloc[:, j].values) for j in range(df.shape[1])]
        )

        # For original subjects use the cached PLV matrix (faster).
        # For augmented copies recompute PLV because the signal has changed.
        base_sid = get_base_sid(sid)
        if sid == base_sid:
            # Original subject (val or test)
            plv = plv_matrices[sid]
        else:
            # Augmented copy (_orig, _noise, _shift) — recompute on modified signal
            plv = generate_plv_matrix(df)

        adj               = plv_to_adj(plv, threshold=fold_threshold)
        adj_matrices[sid] = adj

        # Append graph-structural features: node degree + clustering coefficient
        G           = nx.from_numpy_array(adj)
        deg         = np.array([d for _, d in G.degree()])
        clust       = np.array(list(nx.clustering(G).values()))
        graph_feats = np.vstack([deg, clust]).T

        combined_node_features[sid] = np.concatenate([node_feats, graph_feats], axis=1)

  
    # ── Build per-split dictionaries ──────────────────────────────────────────
    def make_dicts(sids):
        return (
            {s: combined_node_features[s] for s in sids},
            {s: adj_matrices[s]           for s in sids},
            {s: aug_labels[s]             for s in sids},
        )

    # Training uses augmented copies; val and test use originals
    train_nf, train_adj_d, train_lab = make_dicts(aug_train_sids)
    val_nf,   val_adj_d,   val_lab   = make_dicts(val_sids)
    test_nf,  test_adj_d,  test_lab  = make_dicts(test_sids)

    # ==========================================================================
    # TRAINING
    # ==========================================================================
    # ==========================================================================
    # TRAINING avec barre de progression
    # ==========================================================================
    metrics_val, model, scaler, epoch_info, y_true_val, y_pred_val, y_prob_val = \
        train_and_evaluate_gcn(
            train_nf, train_adj_d, train_lab,
            val_nf,   val_adj_d,   val_lab,
            fold_num=fold+1,                    # ← Important
            n_epochs=N_EPOCHS, 
            batch_size=BATCH_SIZE, 
            lr=LR, 
            class_weight_multiplier=CLASS_WEIGHT_MULTIPLIER,
            min_recall_target=0.75
        )

    # ==========================================================================
    # TEST EVALUATION
    # Use the threshold selected on the validation set (no test information used)
    # ==========================================================================
    device = next(model.parameters()).device
    model.eval()

    test_data_list = build_gcn_data_list(
        list(test_nf.keys()), test_nf, test_adj_d, test_lab, scaler=scaler
    )
    test_loader = DataLoader(test_data_list, batch_size=BATCH_SIZE, shuffle=False)

    y_true_test, y_prob_test = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out   = model(batch)
            probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
            y_prob_test.extend(probs)
            y_true_test.extend(batch.y.cpu().numpy())

    # Threshold from val applied to test (no leakage)
    best_th, _  = select_best_threshold_variant(y_true_val, y_prob_val, min_recall=0.75)
    y_pred_test = (np.array(y_prob_test) >= best_th).astype(int)

    test_metrics = {
        'accuracy':  accuracy_score(y_true_test, y_pred_test),
        'precision': precision_score(y_true_test, y_pred_test, zero_division=0),
        'recall':    recall_score(y_true_test, y_pred_test, zero_division=0),
        'f1':        f1_score(y_true_test, y_pred_test, zero_division=0),
        'roc_auc':   roc_auc_score(y_true_test, y_prob_test)
    }


    all_metrics.append(test_metrics)
    y_true_all.extend(y_true_test)
    y_pred_all.extend(y_pred_test)
    y_prob_all.extend(y_prob_test)

    if test_metrics['f1'] > best_f1:
        best_f1, best_model, best_scaler, best_epoch_info = (
            test_metrics['f1'], model, scaler, epoch_info
        )


# ==============================================================================
# AGGREGATE RESULTS
# ==============================================================================

avg_metrics = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0]}
std_metrics  = {k: np.std ([m[k] for m in all_metrics]) for k in all_metrics[0]}

print(f"\n{'='*80}")
print(f"🎉 FINAL RESULTS — Average Test Metrics (± std):")
for k in avg_metrics:
    print(f"  {k.upper():12}: {avg_metrics[k]:.4f} ± {std_metrics[k]:.4f}")
print(f"{'='*80}")


# ==============================================================================
# PLOTS & SAVE
# ==============================================================================

# ── 1. Confusion matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_true_all, y_pred_all)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Control', 'ADHD'], yticklabels=['Control', 'ADHD'])
plt.title("Confusion Matrix —  (All Folds)")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
cm_path = os.path.join(RESULTS_DIR, "confusion_matrix_global.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"✅ Confusion matrix saved → {cm_path}")

# ── 2. ROC curve ─────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_true_all, y_prob_all)
roc_auc     = auc(fpr, tpr)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC  AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve —  (All Folds)')
plt.legend(loc='lower right')
plt.tight_layout()
roc_path = os.path.join(RESULTS_DIR, "roc_curve_global.png")
plt.savefig(roc_path, dpi=150)
plt.show()
print(f"✅ ROC curve saved        → {roc_path}")

# ── 3. Precision-Recall curve ────────────────────────────────────────────────
prec, rec, _ = precision_recall_curve(y_true_all, y_prob_all)
ap_score     = average_precision_score(y_true_all, y_prob_all)
plt.figure(figsize=(6, 6))
plt.plot(rec, prec, color='green', lw=2, label=f'PR Curve  AP = {ap_score:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve —  (All Folds)')
plt.legend(loc='lower left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
pr_path = os.path.join(RESULTS_DIR, "precision_recall_curve.png")
plt.savefig(pr_path, dpi=150)
plt.show()
print(f"✅ PR curve saved         → {pr_path}")


# ==============================================================================
# SAVE MODELS & METADATA
# ==============================================================================

print(f"\nSaving models and metadata to: {RESULTS_DIR}")

torch.save(best_model.state_dict(), os.path.join(RESULTS_DIR, "best_model_weights.pt"))
joblib.dump(best_scaler,            os.path.join(RESULTS_DIR, "scaler.pkl"))

metadata = {
    "dataset":      "ADHD200",
    "architecture": "ProposedGraphSAGE",
    "n_splits":     N_SPLITS,
    "final_metrics": avg_metrics,
    "final_std":     std_metrics,
    "augmentation": {
        "strategies": ["_orig (no change)", "_noise (Gaussian sigma=0.05)", "_shift (circular shift=10)"],
        "applied_to": "train only — val and test are never augmented",
        "train_size_multiplier": 3
    },
    "hyperparameters": {
        "lr":                      LR,
        "batch_size":              BATCH_SIZE,
        "epochs_max":              N_EPOCHS,
        "class_weight_multiplier": CLASS_WEIGHT_MULTIPLIER,
        "plv_percentile":          PLV_PERCENTILE
    }
}
with open(os.path.join(RESULTS_DIR, "final_results_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

predictions_data = {
    "y_true": [int(x)   for x in y_true_all],
    "y_pred": [int(x)   for x in y_pred_all],
    "y_prob": [float(x) for x in y_prob_all]
}
with open(os.path.join(RESULTS_DIR, "raw_predictions.json"), "w") as f:
    json.dump(predictions_data, f)

print("✅ All results saved successfully!")